# Preprocessing

## 1. Setup

In [21]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/NoteVision')
RAW_DIR = PROJECT_DIR / 'data_raw'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
LOCAL_DIR = Path('/content/primus')
ARCHIVE = RAW_DIR / 'primus.tgz'
EXPECTED_SAMPLES = 87_678
SEED = 42

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Extracción

In [2]:
def count_samples(root):
    n = 0
    for pkg in ('package_aa', 'package_ab'):
        p = root / pkg
        if p.exists():
            n += sum(1 for d in p.iterdir() if d.is_dir() and not d.name.startswith('._'))
    return n

if count_samples(LOCAL_DIR) != EXPECTED_SAMPLES:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    !tar -xzf $ARCHIVE -C $LOCAL_DIR

n = count_samples(LOCAL_DIR)
print('Samples on disk:', n)
assert n == EXPECTED_SAMPLES, 'Incomplete extraction'

Samples on disk: 87678


## 3. Load Index

In [3]:
index_df = pd.read_csv(ARTIFACTS_DIR / 'index_agnostic.csv', dtype=str)
index_df['seq_len'] = index_df['seq_len'].astype(int)
print(index_df.shape, index_df.id.is_unique)
print(index_df.columns.tolist())

(87678, 6) True
['id', 'package', 'source_id', 'group', 'seq_len', 'label']


## 4. Group split

In [14]:
TRAIN_FRAC, VAL_FRAC = 0.80, 0.10            # test = 0.10

group_sizes = index_df.groupby('group').size().sort_index()
rng = np.random.default_rng(SEED)
shuffled = group_sizes.iloc[rng.permutation(len(group_sizes))]
cum_frac = shuffled.cumsum() / shuffled.sum()

group_split = pd.Series(
    np.where(cum_frac <= TRAIN_FRAC, 'train',
             np.where(cum_frac <= TRAIN_FRAC + VAL_FRAC, 'val', 'test')),
    index=shuffled.index,
)

index_df['split'] = index_df['group'].map(group_split)
parts = {s: index_df[index_df.split == s] for s in ('train', 'val', 'test')}

for s, d in parts.items():
    print(f"{s} samples = {len(d)} ({len(d)/len(index_df):.2%})  "
          f"groups = {d.group.nunique()}  mean_len = {d.seq_len.mean():.1f}\n")

train samples = 70138 (79.99%)  groups = 46970  mean_len = 27.5

val samples = 8772 (10.00%)  groups = 5742  mean_len = 27.5

test samples = 8768 (10.00%)  groups = 5865  mean_len = 27.5



## 5. Leakage checks

In [17]:
assert sum(len(d) for d in parts.values()) == len(index_df)
assert index_df.split.notna().all()

for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not set(parts[a].group) & set(parts[b].group), f'group leak {a}/{b}'
    assert not set(parts[a].source_id) & set(parts[b].source_id), f'source leak {a}/{b}'
    assert not set(parts[a].label) & set(parts[b].label), f'identical-sequence leak {a}/{b}'

print('No leakage by group, source piece or identical sequence.\n')

# Tokens in val/test never seen in train (cannot be learned)
train_tokens = {t for l in parts['train'].label for t in l.split()}
for s in ('val', 'test'):
    toks = [t for l in parts[s].label for t in l.split()]
    oov = [t for t in toks if t not in train_tokens]
    print(f"{s}: unseen-token occurrences={len(oov)} of {len(toks)}, distinct={len(set(oov))}")

No leakage by group, source piece or identical sequence.

val: unseen-token occurrences=2 of 240867, distinct=2
test: unseen-token occurrences=7 of 240863, distinct=6


## 6. Save

In [18]:
for s, d in parts.items():
    d.drop(columns='split').to_csv(ARTIFACTS_DIR / f'split_{s}.csv', index=False)

split_summary = {
    'seed': SEED, 'fractions': [TRAIN_FRAC, VAL_FRAC, round(1 - TRAIN_FRAC - VAL_FRAC, 2)],
    'unit': 'leak-free group (union of source piece and identical sequence)',
    **{f'n_{s}': len(d) for s, d in parts.items()},
    **{f'n_groups_{s}': int(d.group.nunique()) for s, d in parts.items()},
}
(ARTIFACTS_DIR / 'split_summary.json').write_text(json.dumps(split_summary, indent=2))
print(split_summary)

{'seed': 42, 'fractions': [0.8, 0.1, 0.1], 'unit': 'leak-free group (union of source piece and identical sequence)', 'n_train': 70138, 'n_val': 8772, 'n_test': 8768, 'n_groups_train': 46970, 'n_groups_val': 5742, 'n_groups_test': 5865}


## 7. Funciones para procesado

In [19]:
IMG_HEIGHT = 128

def load_vocab(vocab_csv):
    """Return (tokens, token_to_id, blank_index). CTC blank = last index."""
    df = pd.read_csv(vocab_csv, dtype=str, keep_default_na=False)
    tokens = df['token'].tolist()
    return tokens, {t: i for i, t in enumerate(tokens)}, len(tokens)

def encode_labels(df, token_to_id, max_len):
    """Dense int32 array [N, max_len] (zero padded) and true lengths [N]."""
    labels = np.zeros((len(df), max_len), dtype=np.int32)
    lengths = np.zeros(len(df), dtype=np.int32)
    for i, seq in enumerate(df['label']):
        ids = [token_to_id[t] for t in seq.split()]
        labels[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return labels, lengths

def decode_ids(ids, tokens):
    return ' '.join(tokens[i] for i in ids)

def image_path(root, package, sid):
    return Path(root) / package / sid / f'{sid}.png'

def compute_image_sizes(df, root):
    """Read only PNG headers (fast). Returns DataFrame [id, width, height]."""
    rows = []
    for sid, pkg in zip(df['id'], df['package']):
        with Image.open(image_path(root, pkg, sid)) as im:
            rows.append((sid, im.width, im.height))
    return pd.DataFrame(rows, columns=['id', 'width', 'height'])

def resized_width(width, height, target_h=IMG_HEIGHT):
    return np.maximum(1, np.round(width * target_h / height)).astype(int)

def ctc_margin(new_widths, labels, lengths, reduction):
    """T - (L + consecutive repeats) per sample; CTC needs this >= 0.
    T uses floor(width / reduction)."""
    valid = np.arange(labels.shape[1])[None, :] < lengths[:, None]
    repeats = ((labels[:, 1:] == labels[:, :-1]) & valid[:, 1:]).sum(axis=1)
    return (new_widths // reduction) - (lengths + repeats)

def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_png(raw, channels=3) # palette (mode P) -> RGB
    gray = tf.image.rgb_to_grayscale(img) # uint8 [H, W, 1]
    h = tf.cast(tf.shape(gray)[0], tf.float32)
    w = tf.cast(tf.shape(gray)[1], tf.float32)
    new_w = tf.maximum(1, tf.cast(tf.round(w * IMG_HEIGHT / h), tf.int32))
    gray = tf.image.resize(tf.cast(gray, tf.float32), (IMG_HEIGHT, new_w),
                           method='bilinear', antialias=True)
    gray = tf.clip_by_value(gray, 0.0, 255.0)
    return 1.0 - gray / 255.0, new_w # ink = 1, background/padding = 0

def build_dataset(df, root, token_to_id, max_label_len, batch_size=32,
                  training=False, seed=42):
    labels, lengths = encode_labels(df, token_to_id, max_label_len)
    paths = [str(image_path(root, p, i)) for p, i in zip(df['package'], df['id'])]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def _map(path, label, length):
        image, width = load_image(path)
        return {'image': image, 'width': width,
                'label': label, 'label_length': length}

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.padded_batch(
        batch_size,
        padded_shapes={'image': [IMG_HEIGHT, None, 1], 'width': [],
                       'label': [max_label_len], 'label_length': []},
        padding_values={'image': tf.constant(0.0), 'width': tf.constant(0, tf.int32),
                        'label': tf.constant(0, tf.int32),
                        'label_length': tf.constant(0, tf.int32)})
    return ds.prefetch(tf.data.AUTOTUNE)

## 8. Vocabulario y tamaño de imagenes

In [24]:
tokens, token_to_id, BLANK_INDEX = load_vocab(ARTIFACTS_DIR / 'vocab_agnostic.csv')
MAX_LABEL_LEN = int(index_df.seq_len.max())
print('Vocab:', len(tokens), '| blank index:', BLANK_INDEX, '| max label len:', MAX_LABEL_LEN, '\n')
assert len(tokens) == 758

sizes_path = ARTIFACTS_DIR / 'image_sizes.csv'
if sizes_path.exists():
    size_df = pd.read_csv(sizes_path, dtype={'id': str})
else:
    size_df = compute_image_sizes(index_df, LOCAL_DIR)
    size_df.to_csv(sizes_path, index=False)

size_df['new_width'] = resized_width(size_df.width, size_df.height)
print(size_df[['width', 'height', 'new_width']].describe())

Vocab: 758 | blank index: 758 | max label len: 75 

              width        height     new_width
count  87678.000000  87678.000000  87678.000000
mean    1317.647574    143.129348   1193.280492
std      338.019215     16.007122    339.833115
min      308.000000     74.000000    285.000000
25%     1084.000000    129.000000    957.000000
50%     1274.000000    144.000000   1143.000000
75%     1481.000000    155.000000   1352.000000
max     2003.000000    228.000000   3463.000000


## 9. CTC Factibilidad

In [25]:
labels_all, lengths_all = encode_labels(index_df, token_to_id, MAX_LABEL_LEN)
nw = index_df[['id']].merge(size_df, on='id', how='left')['new_width'].to_numpy()

for r in (2, 4, 8, 16, 32):
    m = ctc_margin(nw, labels_all, lengths_all, r)
    print(f'reduction {r}: infeasible samples={int((m < 0).sum())} | '
          f'min margin={int(m.min())} | median margin={int(np.median(m))}')

reduction 2: infeasible samples=0 | min margin=135 | median margin=542
reduction 4: infeasible samples=0 | min margin=64 | median margin=256
reduction 8: infeasible samples=0 | min margin=28 | median margin=113
reduction 16: infeasible samples=2 | min margin=-7 | median margin=42
reduction 32: infeasible samples=14612 | min margin=-37 | median margin=7


## 10. Pipeline check vs PIL reference + batch

In [27]:
for sid in random.Random(SEED).sample(list(index_df.id), 5):
    row = index_df[index_df.id == sid].iloc[0]
    path = image_path(LOCAL_DIR, row.package, sid)
    ours, w = load_image(str(path))
    with Image.open(path) as im:
        ref = im.convert('L').resize((int(w), IMG_HEIGHT), Image.BILINEAR)
    ref = 1.0 - np.asarray(ref, dtype=np.float32) / 255.0
    print(sid, tuple(ours.shape), 'mean abs diff vs PIL:',
          round(float(np.abs(ours.numpy()[..., 0] - ref).mean()), 4),
          '| range', float(ours.numpy().min()), float(ours.numpy().max()))

train_ds = build_dataset(parts['train'], LOCAL_DIR, token_to_id, MAX_LABEL_LEN,
                         batch_size=16, training=True, seed=SEED)
batch = next(iter(train_ds))
print('\n', {k: tuple(v.shape) for k, v in batch.items()})
print('\n', 'widths:', batch['width'].numpy())
n0 = int(batch['label_length'][0])
print('\n', decode_ids(batch['label'][0, :n0].numpy(), tokens)[:200])

230002354-1_2_1 (128, 856, 1) mean abs diff vs PIL: 0.0001 | range 0.0 0.7764706611633301
000112302-1_1_1 (128, 952, 1) mean abs diff vs PIL: 0.0001 | range 0.0 0.7882353663444519
000102515-1_1_1 (128, 879, 1) mean abs diff vs PIL: 0.0001 | range 0.0 0.7764706611633301
000137438-1_1_1 (128, 1080, 1) mean abs diff vs PIL: 0.0001 | range 0.0 0.7960784435272217
000130480-1_1_1 (128, 1487, 1) mean abs diff vs PIL: 0.0001 | range 0.0 0.7921568751335144

 {'image': (16, 128, 2002, 1), 'width': (16,), 'label': (16, 75), 'label_length': (16,)}

 widths: [1400  827  881  952  828  806 1206  710  797 2002 1125 1045 1137 1198
 1237  947]

 clef.G-L2 accidental.sharp-L5 accidental.sharp-S3 metersign.C-L3 note.eighth-L3 barline-L1 note.beamedRight1-S2 dot-S2 note.beamedBoth2-S0 note.beamedBoth1-S1 dot-S1 note.beamedLeft2-S0 note.beamedRig


## 11. Save Preprocessing config

In [28]:
prep_cfg = {'img_height': IMG_HEIGHT, 'resize': 'bilinear, antialias, keep aspect ratio',
            'grayscale': 'rgb_to_grayscale after decode_png(channels=3)',
            'normalization': 'ink=1, background=0, range [0,1]',
            'batch_padding': 'zeros to widest image in batch',
            'vocab_size': len(tokens), 'blank_index': BLANK_INDEX,
            'max_label_len': MAX_LABEL_LEN, 'label_padding': 0}
(ARTIFACTS_DIR / 'preprocessing_config.json').write_text(json.dumps(prep_cfg, indent=2))

337

## 12. Metrics

In [29]:
from collections import Counter, defaultdict

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (ref[i-1] != hyp[j-1]))
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + (ref[i-1] != hyp[j-1]):
            ops.append(('hit' if ref[i-1] == hyp[j-1] else 'sub', ref[i-1]))
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            ops.append(('del', ref[i-1]))
            i -= 1
        else:
            ops.append(('ins', None))
            j -= 1
    return d[n][m], ops

def evaluate_sequences(refs, hyps, families):
    tot_ref = tot_err = tot_err_shape = exact = 0
    per_fam = defaultdict(Counter)
    for r, h in zip(refs, hyps):
        dist, ops = align(r, h)
        tot_err += dist
        tot_ref += len(r)
        exact += int(r == h)
        tot_err_shape += align([families[i] for i in r], [families[i] for i in h])[0]
        for op, tok in ops:
            if tok is not None:
                per_fam[families[tok]][op] += 1
    metrics = {'cer': tot_err / tot_ref, 'seq_acc': exact / len(refs),
               'cer_shape': tot_err_shape / tot_ref,
               'n_sequences': len(refs), 'n_tokens': tot_ref}
    rows = []
    for fam, c in per_fam.items():
        support = c['hit'] + c['sub'] + c['del']
        rows.append({'family': fam, 'support': support, 'recall': c['hit'] / support})
    fam_df = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)
    return metrics, fam_df

# Sanity checks
_fam = ['a', 'a', 'b']
assert align([0, 1, 2], [0, 1, 2])[0] == 0
assert align([0, 1, 2], [0, 2])[0] == 1
assert align([0, 1, 2], [0, 1, 1])[0] == 1
_m, _ = evaluate_sequences([[0, 1, 2]], [[0, 0, 2]], _fam)
assert abs(_m['cer'] - 1/3) < 1e-9 and _m['cer_shape'] == 0.0 and _m['seq_acc'] == 0.0
print('Metric sanity checks passed.')

Metric sanity checks passed.


## 13. Trivial Baselines (TRAIN only)

In [31]:
families = [t.split('-')[0] for t in tokens]

def to_ids(label):
    return [token_to_id[t] for t in label.split()]

train_seqs = [to_ids(l) for l in parts['train'].label]

# B1: the single most frequent training sequence, predicted for every image
mc_seq, mc_count = Counter(map(tuple, train_seqs)).most_common(1)[0]
print('Most frequent train sequence appears', mc_count, 'times, length', len(mc_seq))

# B2: per-position most frequent token, up to the median training length
L_med = int(np.median([len(s) for s in train_seqs]))
pos_mode = [Counter(s[p] for s in train_seqs if len(s) > p).most_common(1)[0][0]
            for p in range(L_med)]
print('\nPositional-mode length:', L_med)
print(decode_ids(pos_mode, tokens)[:200])

baselines = {'empty': [], 'most_common_sequence': list(mc_seq), 'positional_mode': pos_mode}

baseline_results = {}
for split in ('val', 'test'):
    refs = [to_ids(l) for l in parts[split].label]
    for name, pred in baselines.items():
        m, _ = evaluate_sequences(refs, [pred] * len(refs), families)
        baseline_results[f'{name}/{split}'] = m

print('\n', pd.DataFrame(baseline_results).T[['cer', 'cer_shape', 'seq_acc']].round(4))

Most frequent train sequence appears 7 times, length 27

Positional-mode length: 27
clef.G-L2 accidental.sharp-L5 accidental.sharp-S3 metersign.C-L3 digit.4-L2 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 barline-L1 bar

                               cer  cer_shape  seq_acc
empty/val                  1.0000     1.0000      0.0
most_common_sequence/val   0.9140     0.8096      0.0
positional_mode/val        0.9075     0.8971      0.0
empty/test                 1.0000     1.0000      0.0
most_common_sequence/test  0.9144     0.8076      0.0
positional_mode/test       0.9053     0.8958      0.0


## 14. Metrics Save

In [32]:
metrics_cfg = {
    'primary': 'cer = sum(token edit distance) / sum(reference tokens), corpus-level, agnostic tokens',
    'secondary': ['seq_acc (exact match)',
                  'cer_shape (family only, ignores position)',
                  'per-family recall from alignment'],
    'monitoring': ['train/val CTC loss'],
    'decoding': 'greedy CTC (blank = last index)',
    'protocol': 'val for model selection; test evaluated once for the final model',
    'baselines': {'positional_mode_length': L_med, 'most_common_sequence_count': int(mc_count)},
    'baseline_results': baseline_results,
}
(ARTIFACTS_DIR / 'metrics_baseline.json').write_text(json.dumps(metrics_cfg, indent=2))

1566